# RoadWatch pothole detectorRuns on Kaggle with **Accelerator: GPU P100** and **Internet: On**.Add both datasets as inputs before running:- `juusos/rdd2022es` (RDD2022ES, dashcam-perspective road damage with a pothole/pothole-deep severity split)- `sabidrahman/pothole-cracks-and-openmanhole` (manhole covers, so the model stops reporting storm drains)Both are CC BY-NC-SA 4.0. Non-commercial, attribution required.Use **Save Version -> Save & Run All** so this runs headless and you can close the tab.

In [ ]:
!pip install -q ultralyticsimport os# Ultralytics writes its settings and a wandb prompt somewhere on first import.os.environ["YOLO_CONFIG_DIR"] = "/kaggle/working"os.environ["WANDB_DISABLED"] = "true"# Public repo, so no credentials needed. If you make it private, upload# prep_dataset.py as a Kaggle dataset and point at that instead.!rm -rf /kaggle/working/roadwatch!git clone -q https://github.com/K-man1/roadwatch.git /kaggle/working/roadwatch

## Build the dataset`/kaggle/input` is read-only and Ultralytics writes a `labels.cache` next to the labelsdirectory on the first epoch, so the dataset has to live somewhere writable. `/kaggle/temp`is the right choice over `/kaggle/working`: both are writable, but working gets versionedinto the notebook output on every commit and you do not want to re-upload gigabytes ofimages each run.The prep script drops RDD2022ES's mirrored duplicate frames (Ultralytics already applies`fliplr` during training, so keeping both halves doubles the epoch for nothing), keeps onlythe two pothole tiers, folds in pothole-free manhole frames, and caps background frames at10% of the training set.Add `--countries United_States Czech` to test whether the Japanese and Indian road surfaceshelp or hurt on New Jersey footage.

In [ ]:
!rm -rf /kaggle/temp/data!python /kaggle/working/roadwatch/prep_dataset.py \    --rdd /kaggle/input/rdd2022es/combined_annotatedv2 \    --manhole /kaggle/input/pothole-cracks-and-openmanhole \    --out /kaggle/temp/dataDATA = "/kaggle/temp/data/data.yaml"NAMES = ["pothole", "pothole_deep", "manhole"]print(open(DATA).read())

## Check the labels before you spend GPU hours on themA silently wrong class remap looks exactly like a correct one until you plot it. Boxesshould sit on actual road damage, and `pothole_deep` should look meaningfully worse than`pothole`. If it does not, the severity split is not worth training as two classes.

In [ ]:
import randomfrom pathlib import Pathimport matplotlib.patches as patchesimport matplotlib.pyplot as pltfrom PIL import ImageCOLORS = ["#ff6b35", "#d7263d", "#2e86ab"]def label_for(image_path):    return Path(str(image_path).replace("/images/", "/labels/")).with_suffix(".txt")annotated = [p for p in sorted(Path("/kaggle/temp/data/images/train").iterdir())             if label_for(p).read_text().strip()]sample = random.Random(0).sample(annotated, 8)fig, axes = plt.subplots(2, 4, figsize=(20, 9))for ax, path in zip(axes.ravel(), sample):    image = Image.open(path)    width, height = image.size    ax.imshow(image)    ax.axis("off")    for line in label_for(path).read_text().splitlines():        cls, x, y, w, h = line.split()[:5]        cls = int(cls)        x, y, w, h = float(x) * width, float(y) * height, float(w) * width, float(h) * height        ax.add_patch(patches.Rectangle((x - w / 2, y - h / 2), w, h,                                       fill=False, lw=2, edgecolor=COLORS[cls]))        ax.text(x - w / 2, y - h / 2 - 4, NAMES[cls], color=COLORS[cls], fontsize=9)plt.tight_layout()plt.show()

## Train`imgsz=960` rather than the usual 640 is the deliberate choice here. A pothole thirty metresdown the road is around 40px tall in a 1080p frame; at 640 that lands near YOLO's stride-8detection floor and the model simply cannot see it. More input resolution beats moreparameters when the objects are small and far away. It costs phone inference time, sobenchmark before committing.`batch=-1` is AutoBatch, which probes VRAM and targets about 60% use. If it OOMs mid-run,pin it to 16.Expect roughly 45 to 90 seconds per epoch on a P100, so 2 to 3 hours. Well inside the 12hour session cap, and you get 30 GPU hours a week.

In [ ]:
from ultralytics import YOLOmodel = YOLO("yolo11s.pt")model.train(    data=DATA,    epochs=150,    imgsz=960,    batch=-1,    device=0,    workers=4,    patience=30,    seed=0,    project="/kaggle/working/runs",    name="roadwatch",)

## Per-class metrics on the held-out test splitOne overall mAP number tells you nothing about which class is dragging. Read theseseparately: if `manhole` is strong and `pothole_deep` is weak, the severity split is theproblem, not the detector.

In [ ]:
metrics = model.val(data=DATA, split="test")print(f"{'class':14s} {'P':>7s} {'R':>7s} {'mAP50':>7s} {'mAP50-95':>9s}")for i, name in enumerate(NAMES):    precision, recall, ap50, ap = metrics.box.class_result(i)    print(f"{name:14s} {precision:7.3f} {recall:7.3f} {ap50:7.3f} {ap:9.3f}")

## Pick the auto-report confidence thresholdThis is a product decision, not a metric. Filing a false report with the town is far morecostly than missing one pothole, so the threshold that maximises F1 is probably not the oneyou want. Read the precision column and pick the lowest confidence that still keepsprecision high enough that you would defend the report.Each row is a full validation pass, so this cell takes a few minutes.

In [ ]:
import pandas as pdrows = []for conf in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7]:    swept = model.val(data=DATA, split="val", conf=conf, plots=False, verbose=False)    precision, recall, _, _ = swept.box.class_result(NAMES.index("pothole_deep"))    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0    rows.append({"conf": conf, "precision": precision, "recall": recall, "f1": f1})print(pd.DataFrame(rows).to_string(index=False))

## Export for the phonencnn is the runtime you were already targeting. Export at the resolution you intend to runat: exporting at 960 and then feeding it 640 frames throws away the reason you trained at960 in the first place, but 960 on a phone may not hold 30fps. Benchmark both.`best.pt` is copied to the notebook output root so you can download it without diggingthrough the runs directory.

In [ ]:
import shutilbest = "/kaggle/working/runs/roadwatch/weights/best.pt"YOLO(best).export(format="ncnn", imgsz=960)shutil.copy(best, "/kaggle/working/roadwatch_best.pt")print("done")